In [ ]:
# %pip install -U langchain-google-genai
# %pip install dotenv

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load = load_dotenv('../.env')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

tc_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
)

In [ ]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
import os
from langchain_community.document_loaders import (
    UnstructuredPDFLoader
)


@tool
def generate_test_cases_from_pdf() -> str:
    "Reads the requirements from the PDF and generate atleast 10 test scenarios in BDD Format"
    
    docs_folder = "./Docs"
    documents = []

    for filename in os.listdir(docs_folder):
        filepath = os.path.join(docs_folder, filename)
        loader = UnstructuredPDFLoader(filepath)
        docs = loader.load();
        for doc in docs:
            documents.append(f"{filename}:\n{doc.page_content.strip()}")
    
    requirement_txt = "\n\n".join(documents[:3])[:1000]
    
   
    prompt_template = PromptTemplate.from_template(
        
        """
        You are are QA Automation Engineer.
        Your task is to convert the following user stories into at least 10 testcases, in Gherkin BDD style format.
        Include combinations of valid, invalid, edge case and alternative flow scearios
        
        {requirement_txt}
        
        Format:
        Feature: ...
        Scenario: ...
            Given ...
            When ...
            Then ...
        
        """
    )
    prompt = prompt_template.invoke({"requirement_txt":requirement_txt})
    return tc_llm.invoke(prompt)
    

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools= [generate_test_cases_from_pdf],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

result = agent.invoke("Generate the test cases from PDF file by reading the requirement pdf documents")

print(result["output"])
